# 03 — Deep Learning Regression

Experiment with a neural network regressor before finalizing `src/train_dl.py`.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.train_dl import load_processed, build_model

tf.random.set_seed(42)

In [ ]:
X_train, X_test, y_train, y_test = load_processed('../data/processed')
y_mean, y_std = y_train.mean(), y_train.std()
y_train_scaled = (y_train - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std
X_train.shape

## Build and train the model

In [ ]:
model = build_model(input_dim=X_train.shape[1])
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train, y_train_scaled,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

## Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss (MSE, scaled target)')
ax[0].legend()

ax[1].plot(history.history['mae'], label='train')
ax[1].plot(history.history['val_mae'], label='val')
ax[1].set_title('MAE (scaled target)')
ax[1].legend()
plt.tight_layout()
plt.show()

## Evaluate on test set

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

preds_scaled = model.predict(X_test).flatten()
preds = preds_scaled * y_std + y_mean

rmse = mean_squared_error(y_test, preds, squared=False)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f'RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.3f}')

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, preds, alpha=0.4)
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual price')
plt.ylabel('Predicted price')
plt.title('DL model: actual vs predicted')
plt.show()

## Next steps

Tune architecture/hyperparameters here, then port the final configuration into `src/train_dl.py` for reproducible training and to compare against the ML models via `src/evaluate.py`.